# Value Iteration FPGA — Real Map Exploration

Ultra96-V2 の PYNQ 上で VI アクセラレータを使い、ROS map_server 互換 PGM+YAML (例: `map_tsudanuma`) を読み込んで実際にプランニングとロールアウトまで行う notebook。

## 使い方

1. 同ディレクトリに `vi_bd_wrapper.bit` / `vi_bd_wrapper.hwh` / `vi_overlay.py` があることを確認。
2. マップファイル (`.pgm` + `.yaml`) を配置し、下の `MAP_YAML` を書き換える。
3. `START_XY` / `GOAL_XY` をワールド座標 (m) で指定 (YAML の `origin` を考慮される)。
4. 順にセルを実行すると `./out/` に結果画像と `result.npz` / `sweep_history.csv` が保存される。

## 出力

- `out/sweep_history.csv` — sweep 毎の `max_delta` と所要時間
- `out/convergence.png` — 収束曲線
- `out/path.png` — penalty マップに start→goal 経路を重ねた画像
- `out/result.npz` — `value` / `action_table` / `penalty` / `path`

In [ ]:
# === Parameters ===
# --- Matches Ueda et al. 2023 (JRM), Tsudanuma actual experiment ---
# See Table 1 (actions) and Table 2 "Actual" column.
from pathlib import Path

MAP_YAML          = "./maps/map_tsudanuma.yaml"
TARGET_RESOLUTION = 0.15            # [m]  paper Table 2 "Actual"
START_XY          = (-80.0, -50.0)  # world meters (rewrite as needed)
GOAL_XY           = (-60.0, -40.0)  # world meters
START_THETA_DEG   = 0.0
GOAL_RADIUS_M     = 0.3             # paper: 300 mm goal area radius
SAFETY_RADIUS_M   = 0.2             # paper: nnear = 200 mm robot radius
THRESHOLD         = 0               # max_delta がこれ以下になったら収束
MAX_SWEEPS        = 500
BITSTREAM         = "vi_bd_wrapper.bit"
OUT_DIR           = Path("./out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# === Imports & map loader ===
import math
import time
import numpy as np
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import binary_dilation

N_ACTIONS = 6
N_THETA   = 60
TILE_W    = 32
TILE_H    = 32
MAX_VALUE        = 0xFFFF
PENALTY_OBSTACLE = 0xFFFF
PENALTY_GOAL     = 0xFFFE  # HLS kernel treats this as 0 when read as neighbor

def load_pgm_map(yaml_path):
    yaml_path = Path(yaml_path)
    with open(yaml_path) as f:
        meta = yaml.safe_load(f)
    img_path = (yaml_path.parent / meta["image"]).resolve()
    img = np.array(Image.open(img_path))
    if meta.get("negate", 0):
        img = 255 - img
    # Flip so row 0 = bottom (ROS map convention, y increases upward).
    img = np.flipud(img)
    return {
        "pixels": img,
        "resolution": float(meta["resolution"]),
        "origin_x": float(meta["origin"][0]),
        "origin_y": float(meta["origin"][1]),
        "occupied_thresh": float(meta["occupied_thresh"]),
        "free_thresh": float(meta["free_thresh"]),
    }

m = load_pgm_map(MAP_YAML)
H, W = m["pixels"].shape
res  = m["resolution"]
print(f"Map: {W} x {H}, resolution={res}m, origin=({m['origin_x']}, {m['origin_y']})")

In [ ]:
# === Penalty table (downsample to target resolution + inflate obstacles) ===
pix = m["pixels"]
occ_prob = (255 - pix.astype(np.float32)) / 255.0
occupied_src = occ_prob >= m["occupied_thresh"]

# --- Downsample from native YAML resolution to TARGET_RESOLUTION ---
# Conservative max-pool: a coarse cell is occupied if ANY fine cell inside is.
scale = int(round(TARGET_RESOLUTION / m["resolution"]))
assert scale >= 1, f"target resolution {TARGET_RESOLUTION} < map resolution {m['resolution']}"
if scale > 1:
    Hc = (H // scale) * scale
    Wc = (W // scale) * scale
    occ_trim = occupied_src[:Hc, :Wc]
    occupied = occ_trim.reshape(Hc // scale, scale, Wc // scale, scale).any(axis=(1, 3))
else:
    occupied = occupied_src
res = m["resolution"] * scale  # effective cell size, e.g. 0.15 m
H2, W2 = occupied.shape
print(f"Downsampled: {W2} x {H2} cells at {res}m (scale={scale})")

# --- Inflate by safety radius (paper: nnear = 200 mm) ---
radius_cells = int(math.ceil(SAFETY_RADIUS_M / res))
yy, xx = np.ogrid[-radius_cells:radius_cells + 1, -radius_cells:radius_cells + 1]
disk = (xx**2 + yy**2) <= radius_cells**2
inflated = binary_dilation(occupied, structure=disk)

def world_to_grid(wx, wy):
    mx = int(math.floor((wx - m["origin_x"]) / res))
    my = int(math.floor((wy - m["origin_y"]) / res))
    return mx, my

goal_mx,  goal_my  = world_to_grid(*GOAL_XY)
start_mx, start_my = world_to_grid(*START_XY)
assert 0 <= goal_mx  < W2 and 0 <= goal_my  < H2, f"goal out of map: ({goal_mx},{goal_my})"
assert 0 <= start_mx < W2 and 0 <= start_my < H2, f"start out of map: ({start_mx},{start_my})"
if inflated[goal_my, goal_mx]:
    print(f"WARNING: goal cell ({goal_mx},{goal_my}) lies in inflated obstacle")
if inflated[start_my, start_mx]:
    print(f"WARNING: start cell ({start_mx},{start_my}) lies in inflated obstacle")

# --- Pad to TILE_W/TILE_H multiples (extra cells treated as obstacles) ---
pad_h = (TILE_H - H2 % TILE_H) % TILE_H
pad_w = (TILE_W - W2 % TILE_W) % TILE_W
if pad_h or pad_w:
    inflated = np.pad(inflated, ((0, pad_h), (0, pad_w)), constant_values=True)
MAP_Y, MAP_X = inflated.shape
print(f"Padded map: {MAP_X} x {MAP_Y} (pad_w={pad_w}, pad_h={pad_h})")

# --- Goal region: disk of GOAL_RADIUS_M around GOAL_XY (paper: 300 mm) ---
goal_radius_cells = int(math.ceil(GOAL_RADIUS_M / res))
gy, gx = np.ogrid[0:MAP_Y, 0:MAP_X]
goal_mask = ((gx - goal_mx) ** 2 + (gy - goal_my) ** 2) <= goal_radius_cells ** 2
# Goal cells must be reachable even if they overlap the inflation halo.
goal_mask &= ~np.pad(
    np.zeros_like(occupied, dtype=bool) | occupied,
    ((0, pad_h), (0, pad_w)),
    constant_values=True,
)

penalty = np.where(inflated, PENALTY_OBSTACLE, 0).astype(np.uint16)
penalty[goal_mask] = PENALTY_GOAL
n_goal_cells = int(goal_mask.sum())
assert n_goal_cells > 0, "goal region empty (goal inside an obstacle?)"
print(f"Start grid=({start_mx},{start_my}), Goal center=({goal_mx},{goal_my}), goal cells={n_goal_cells}")

In [ ]:
# === Value table init & transition model ===
# Actions per Ueda et al. 2023 Table 1 (left/right forward: v = 0.2 m/s).
value = np.full((MAP_Y, MAP_X, N_THETA), MAX_VALUE, dtype=np.uint16)
value[goal_mask] = 0

ACTION_FW  = [0.3, -0.2, 0.0,  0.0, 0.2,  0.2]
ACTION_ROT = [0.0,  0.0, 20.0, -20.0, 20.0, -20.0]

def compute_transitions(xy_res):
    t_res  = 360.0 / N_THETA
    packed = np.zeros(N_ACTIONS * N_THETA, dtype=np.uint32)
    deltas = np.zeros((N_ACTIONS, N_THETA, 3), dtype=np.int32)
    for a in range(N_ACTIONS):
        for it in range(N_THETA):
            theta = (it * t_res + t_res * 0.5) * math.pi / 180.0
            dx = ACTION_FW[a] * math.cos(theta)
            dy = ACTION_FW[a] * math.sin(theta)
            dix = int(math.floor(dx / xy_res))
            diy = int(math.floor(dy / xy_res))
            new_theta = it * t_res + t_res * 0.5 + ACTION_ROT[a]
            while new_theta < 0:     new_theta += 360
            while new_theta >= 360:  new_theta -= 360
            new_it = int(math.floor(new_theta / t_res))
            dit = new_it - it
            if dit >  N_THETA // 2: dit -= N_THETA
            if dit < -N_THETA // 2: dit += N_THETA
            w = (dix & 0xFF) | ((diy & 0xFF) << 8) | ((dit & 0xFF) << 16)
            packed[a * N_THETA + it] = w
            deltas[a, it] = (dix, diy, dit)
    return packed, deltas

trans, deltas = compute_transitions(res)
print(f"Transitions: {trans.shape}, value buffer: {value.shape} ({value.nbytes/1e6:.1f} MB)")

In [ ]:
# === FPGA sweep loop with per-sweep benchmarking ===
# Reimplements VIOverlay.run() inline so we can record max_delta and
# elapsed time for each individual sweep.
from pynq import Overlay, allocate
from tqdm.auto import tqdm
from vi_overlay import (
    AP_CTRL, ADDR_VALUE_TABLE, ADDR_PENALTY, ADDR_TRANS,
    ADDR_MAP_X, ADDR_MAP_Y, ADDR_NUM_TILES_X, ADDR_NUM_TILES_Y,
    ADDR_CU_ID, ADDR_MAX_DELTA, _write_addr64,
)

print("Loading overlay...")
ol  = Overlay(BITSTREAM)
cu0 = ol.vi_sweep_cu0
cu1 = ol.vi_sweep_cu1

val_buf = allocate(shape=value.shape,   dtype=np.uint16)
pen_buf = allocate(shape=penalty.shape, dtype=np.uint16)
trn_buf = allocate(shape=trans.shape,   dtype=np.uint32)
np.copyto(val_buf, value)
np.copyto(pen_buf, penalty)
np.copyto(trn_buf, trans)
val_buf.sync_to_device()
pen_buf.sync_to_device()
trn_buf.sync_to_device()

num_tiles_x = MAP_X // TILE_W
num_tiles_y = MAP_Y // TILE_H
for cu in (cu0, cu1):
    _write_addr64(cu, ADDR_VALUE_TABLE, val_buf.device_address)
    _write_addr64(cu, ADDR_PENALTY,     pen_buf.device_address)
    _write_addr64(cu, ADDR_TRANS,       trn_buf.device_address)
    cu.write(ADDR_MAP_X,       MAP_X)
    cu.write(ADDR_MAP_Y,       MAP_Y)
    cu.write(ADDR_NUM_TILES_X, num_tiles_x)
    cu.write(ADDR_NUM_TILES_Y, num_tiles_y)

history = []
t_start = time.time()
pbar = tqdm(range(MAX_SWEEPS), desc="VI sweeps", unit="sweep")
for sweep in pbar:
    t0 = time.time()
    cu0.write(ADDR_CU_ID, 0); cu1.write(ADDR_CU_ID, 1)
    cu0.write(AP_CTRL, 0x01); cu1.write(AP_CTRL, 0x01)
    while not (cu0.read(AP_CTRL) & 0x02): pass
    while not (cu1.read(AP_CTRL) & 0x02): pass
    d0 = cu0.read(ADDR_MAX_DELTA)
    d1 = cu1.read(ADDR_MAX_DELTA)
    max_delta = max(d0, d1)
    dt_ms = (time.time() - t0) * 1e3
    history.append({"sweep": sweep, "max_delta": int(max_delta), "dt_ms": dt_ms})
    pbar.set_postfix(max_delta=int(max_delta), dt_ms=f"{dt_ms:.1f}")
    if max_delta <= THRESHOLD:
        pbar.close()
        break
else:
    pbar.close()
elapsed = time.time() - t_start

val_buf.sync_from_device()
np.copyto(value, val_buf)
val_buf.freebuffer(); pen_buf.freebuffer(); trn_buf.freebuffer()

converged = history[-1]["max_delta"] <= THRESHOLD
print(
    f"converged={converged}  sweeps={len(history)}  "
    f"elapsed={elapsed:.3f}s  final_delta={history[-1]['max_delta']}"
)

hist_df = pd.DataFrame(history)
hist_df.to_csv(OUT_DIR / "sweep_history.csv", index=False)
hist_df.tail()

In [ ]:
# === Action table (ARM-side argmin over actions) ===
# For each state (mx, my, it) pick the action minimizing V(s') + penalty(s').
# Goal penalty (PENALTY_GOAL) is treated as 0 when read as a neighbor,
# matching the HLS kernel convention.
pen_eff = np.where(penalty == PENALTY_GOAL, 0, penalty).astype(np.int32)
val32   = value.astype(np.int32)

INF = np.iinfo(np.int32).max
action_table = np.zeros((MAP_Y, MAP_X, N_THETA), dtype=np.uint8)
best_cost    = np.full((MAP_Y, MAP_X, N_THETA), INF, dtype=np.int32)

yy, xx = np.meshgrid(np.arange(MAP_Y), np.arange(MAP_X), indexing="ij")

for a in range(N_ACTIONS):
    for it in range(N_THETA):
        dix, diy, dit = (int(v) for v in deltas[a, it])
        nx = xx + dix
        ny = yy + diy
        nt = (it + dit) % N_THETA
        in_bounds = (nx >= 0) & (nx < MAP_X) & (ny >= 0) & (ny < MAP_Y)
        nx_c = np.clip(nx, 0, MAP_X - 1)
        ny_c = np.clip(ny, 0, MAP_Y - 1)
        v_next = val32[ny_c, nx_c, nt]
        p_next = pen_eff[ny_c, nx_c]
        cost = np.where(in_bounds, v_next + p_next, INF)
        better = cost < best_cost[:, :, it]
        best_cost[:, :, it]    = np.where(better, cost, best_cost[:, :, it])
        action_table[:, :, it] = np.where(better, a,    action_table[:, :, it])

print("Action table computed.")

In [ ]:
# === Rollout from start to goal following the action table ===
start_it = int((START_THETA_DEG % 360) / (360.0 / N_THETA))
ix, iy, it = start_mx, start_my, start_it
path = [(ix, iy, it)]
MAX_STEPS = 20000
reached = False
stop_reason = "max_steps"
for step in range(MAX_STEPS):
    if goal_mask[iy, ix]:
        reached = True
        stop_reason = "goal"
        break
    a = int(action_table[iy, ix, it])
    dix, diy, dit = (int(v) for v in deltas[a, it])
    nix = ix + dix
    niy = iy + diy
    nit = (it + dit) % N_THETA
    if not (0 <= nix < MAP_X and 0 <= niy < MAP_Y):
        stop_reason = "out_of_bounds"
        break
    if penalty[niy, nix] == PENALTY_OBSTACLE:
        stop_reason = "obstacle"
        break
    if (nix, niy, nit) == (ix, iy, it):
        stop_reason = "stuck"
        break
    ix, iy, it = nix, niy, nit
    path.append((ix, iy, it))

path_arr = np.array(path, dtype=np.int32)
print(f"Rollout: reached={reached}  steps={len(path) - 1}  stop={stop_reason}")

In [ ]:
# === Visualization & save ===
aspect = max(3.0, 14.0 * MAP_Y / MAP_X)
fig, ax = plt.subplots(figsize=(14, aspect))
ax.imshow(penalty, origin="lower", cmap="gray_r", vmin=0, vmax=PENALTY_OBSTACLE)
ax.plot(path_arr[:, 0], path_arr[:, 1], "r-", linewidth=1.5, label="path")
ax.plot(start_mx, start_my, "go", markersize=8,  label="start")
ax.plot(goal_mx,  goal_my,  "b*", markersize=12, label="goal")
ax.set_title(f"VI FPGA exploration \u2014 {Path(MAP_YAML).stem}")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(OUT_DIR / "path.png", dpi=120)
plt.show()

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(hist_df["sweep"], hist_df["max_delta"].clip(lower=1))
ax2.set_xlabel("sweep")
ax2.set_ylabel("max_delta (clipped at 1 for log)")
ax2.set_yscale("log")
ax2.set_title(f"Convergence ({len(hist_df)} sweeps, {elapsed:.2f}s)")
fig2.tight_layout()
fig2.savefig(OUT_DIR / "convergence.png", dpi=120)
plt.show()

np.savez_compressed(
    OUT_DIR / "result.npz",
    value=value,
    action_table=action_table,
    penalty=penalty,
    path=path_arr,
    start=np.array([start_mx, start_my, start_it], dtype=np.int32),
    goal=np.array([goal_mx,  goal_my],              dtype=np.int32),
)
print(f"Saved results to {OUT_DIR.resolve()}")
print(f"  sweeps={len(hist_df)}  elapsed={elapsed:.3f}s  reached={reached}  stop={stop_reason}")